# Taymar Walters' Final Project – Breast Cancer Classification
Complete Jupyter Notebook


## 1. Imports and Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc, brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import os

print("Libraries loaded successfully.")


## 2. Load and Explore Dataset


In [ ]:
data = load_breast_cancer()
X = data.data
y_orig = data.target

# Remap to binary: malignant=1, benign=0
y = (y_orig == 0).astype(int)

df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y

df.head()


### Dataset Summary


In [ ]:
print("Shape:", df.shape)
df['target'].value_counts()


## 3. Helper Metric Functions


In [ ]:
def safe_div(a, b):
    return a/b if b != 0 else np.nan

def compute_counts(y_true, y_pred):
    TN, FP, FN, TP = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return TP, TN, FP, FN, TP+FN, TN+FP

def compute_skill_scores(TP, TN, FP, FN):
    P = TP+FN; N = TN+FP
    TPR = safe_div(TP,P)
    FPR = safe_div(FP,N)
    TSS = TPR - FPR
    denom = (TP+FN)*(FN+TN) + (TP+FP)*(FP+TN)
    HSS = safe_div(2*(TP*TN - FP*FN), denom)
    return TSS, HSS

def compute_fold_metrics(y_true, y_pred, y_prob):
    TP, TN, FP, FN, P, N = compute_counts(y_true, y_pred)
    TPR = safe_div(TP,P); TNR = safe_div(TN,N)
    FPR = safe_div(FP,N); FNR = safe_div(FN,P)
    acc = safe_div(TP+TN, P+N)
    bal_acc = np.nanmean([TPR, TNR])
    prec = safe_div(TP, TP+FP)
    rec = TPR
    f1 = safe_div(2*prec*rec, prec+rec)
    err = 1 - acc
    TSS, HSS = compute_skill_scores(TP,TN,FP,FN)
    bs = brier_score_loss(y_true, y_prob)
    p_bar = np.mean(y_true)
    bs_ref = p_bar * (1 - p_bar)
    bss = 1 - safe_div(bs, bs_ref)

    return {
        'TP':TP, 'TN':TN, 'FP':FP, 'FN':FN, 'P':P, 'N':N,
        'TPR':TPR, 'TNR':TNR, 'FPR':FPR, 'FNR':FNR,
        'Accuracy':acc, 'BalancedAccuracy':bal_acc,
        'Precision':prec, 'Recall':rec, 'F1':f1, 'ErrorRate':err,
        'TSS':TSS, 'HSS':HSS, 'BS':bs, 'BSS':bss
    }


## 4. Define Models


In [ ]:
models = {
    'LogisticRegression': Pipeline([
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(max_iter=5000))
    ]),
    'SVM_RBF': Pipeline([
        ('scale', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True))
    ]),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'MLP': Pipeline([
        ('scale', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=(64,32), max_iter=2000, random_state=42))
    ])
}

print("Models initialized.")


## 5. 10-Fold Cross-Validation


In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = {}
roc_data = {}

for model_name, model in models.items():
    fold_metrics = []
    all_true = []; all_prob = []

    print(f"Training model: {model_name}")

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else y_pred.astype(float)

        metrics = compute_fold_metrics(y_test, y_pred, y_prob)
        metrics['Fold'] = fold
        fold_metrics.append(metrics)

        all_true.append(y_test)
        all_prob.append(y_prob)

    df_model = pd.DataFrame(fold_metrics)
    results[model_name] = df_model

    all_true = np.concatenate(all_true)
    all_prob = np.concatenate(all_prob)
    fpr, tpr, _ = roc_curve(all_true, all_prob)
    roc_data[model_name] = (fpr, tpr, auc(fpr, tpr))

results


## 6. ROC Curves


In [ ]:
for model_name, (fpr, tpr, model_auc) in roc_data.items():
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {model_auc:.4f}")
    plt.plot([0,1], [0,1], 'k--')
    plt.title(f"ROC Curve – {model_name}")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()


## 7. Save Results


In [ ]:
os.makedirs('results_dm_final', exist_ok=True)

for model_name, df in results.items():
    df.to_csv(f"results_dm_final/{model_name}_10fold_metrics.csv", index=False)

print("All results saved to /results_dm_final")
